## Student Attendance Analytics with Pandas

In [ ]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("CSEB_Attendance_Synthetic.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
#"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
print(df.head(10))
print(df.shape)
print(df.dtypes)

In [ ]:
# Q2. Check for missing values in each column.
# Identify columns with null values and report the count.

df['Last Present Date'] = df['Last Present Date'].replace('Never', pd.NA)
print(df.isnull().sum())

In [ ]:
#"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
df.columns = [
    'roll_no',
    'student_name',
    'subject',
    'total_classes',
    'present',
    'absent',
    'excused',
    'attendance_pct',
    'status',
    'classes_needed_75_pct',
    'last_present_date',
    'consecutive_absences'
]

print(df.columns)

In [ ]:
#"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
df['attendance_pct'] = df['attendance_pct'].str.replace('%', '').astype(float)

df['attendance_pct'] = df['attendance_pct'] / 100

df['last_present_date'] = pd.to_datetime(
    df['last_present_date'].replace('Never', np.nan)
)

print(df)

#### Section 2: Filtering & Selection

In [ ]:
#"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
result = df[df['status'] == 'Critical'].sort_values('attendance_pct')

print(result)

In [ ]:
print(df.columns.tolist())

In [ ]:
#"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
print(df[df['present'] == 0][
    ['roll_no', 'student_name', 'consecutive_absences']
])

In [ ]:
#"""Q7. Using .loc and .iloc, display:
#(a) All "Safe" students using .loc
#(b) The first 15 rows and columns 2-6 using .iloc """
print(df.loc[df['status'] == 'Safe'])
print(df.iloc[:15, 1:6])

In [ ]:
#"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
result = df[df['consecutive_absences'] >= 5]

print(result[['student_name', 'status', 'last_present_date']])

### Section 3: Grouping & Aggregation 

In [ ]:
#"""Q9. Group the data by Status and compute: 
#a) Count of students 
#b) Average attendance %  
#c) Maximum consecutive absences"""
result = df.groupby('status').agg({
    'student_name': 'count',
    'attendance_pct': 'mean',
    'consecutive_absences': 'max'
})

print(result)

In [ ]:
#"""Q10. Create a new column Attendance Category:
#≥ 90% → "Excellent"
#75-89% → "Good"
#60-74% → "Average"
#< 60% → "Poor"
#Then use groupby() to count students in each category."""
df['attendance_category'] = pd.cut(
    df['attendance_pct'],
    bins=[0, 0.60, 0.75, 0.85, 1.0],
    labels=['Poor', 'Average', 'Good', 'Excellent']
)

print(df)

In [ ]:
#"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
print("Total Classes:", df['total_classes'].sum())
print("Total Present:", df['present'].sum())
print("Total Absent:", df['absent'].sum())

In [ ]:
#"""Q12. Group by Status and use .agg() to find:
#a)min, max, and mean of Present
#b)sum of Absent"""
result = df.groupby('status').agg({
    'attendance_pct': ['min', 'max']
})

print(result)

### Section 4: Feature Engineering & Transformation

In [ ]:
#"""Q13. Create a new column Risk Score using the formula:
#Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
#Display the top 10 students with the highest risk score."""
df['risk_score'] = (
    df['absent'] * 0.4 +
    df['consecutive_absences'] * 0.6
)

print(df.nlargest(10, 'risk_score'))

In [ ]:
#"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
df['classes_needed'] = pd.to_numeric(
    df['classes_needed_75_pct'].astype(str).str.extract(r'(\d+)')[0],
    errors='coerce'
).fillna(0).astype(int)

print(df[['student_name', 'classes_needed']])

In [ ]:
#"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
#a)"Defaulter" if attendance < 75%
#b)"Safe" otherwise """
df['attendance_flag'] = df['attendance_pct'].apply(
    lambda x: 'Defaulter' if x < 75 else 'Safe'
)

print(df[['student_name', 'attendance_flag']])

In [ ]:
#"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
df['attendance_range'] = pd.cut(
    df['attendance_pct'],
    bins=[0, 50, 75, 90, 100],
    labels=['Critical', 'Warning', 'Good', 'Excellent'],
    include_lowest=True
)

print(df[['student_name', 'attendance_range']])

### Section 5: Advanced Analysis 

In [ ]:
#"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
result = pd.pivot_table(
    df,
    index='status',
    columns='attendance_category',
    values='student_name',
    aggfunc='count',
    fill_value=0
)

print(result)

In [ ]:
#"""Q18. Identify duplicate student names (e.g., "STUDENT A", "STUDENT B") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
df['base_name'] = df['student_name'].str.split().str[0]

duplicates = df[df.duplicated('base_name', keep=False)]

print(duplicates)

In [ ]:
#""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
student_details = pd.DataFrame({
    'roll_no': df['roll_no'],
    'branch': ['CSE'] * len(df),
    'semester': [5] * len(df)
})

result = pd.merge(df, student_details, on='roll_no', how='inner')

print(result)

In [ ]:
#"""Q20. Export the following to separate CSV files:
#(a) All "Critical" students
#(b) Summary statistics grouped by Status
#(c) Top 10 students by Risk Score"""
df[df['status'] == 'Critical'].to_csv(
    'critical_students.csv', index=False
)
df.groupby('status').describe().to_csv('summary_statistics.csv')
df.nlargest(10, 'risk_score').to_csv(
    'top10_risk_students.csv', index=False
)
df
